### Update to the basic app to generate insights

In [39]:
!pip install gradio boto3 langchain_community

In [40]:
import os
import gradio as gr
import pandas as pd
from langchain_community.document_loaders import DataFrameLoader
import json
import boto3
from botocore.exceptions import ClientError

In [ ]:
import os

DATA_FOLDER = os.getcwd().split("genai_solution")[0] + "genai_solution\\data\\"


In [42]:

total = pd.read_csv(DATA_FOLDER + "monthly_totals.csv")
product_data = pd.read_csv(DATA_FOLDER + "monthly_product_totals.csv")

month_list = list(product_data["order_month"].unique())

In [44]:
# Defining a function to get a llm response from aws
# This code is taken from above links


def get_output_aws(prompt):
    client = boto3.client("bedrock-runtime", region_name="eu-west-1")

    model_id = "eu.anthropic.claude-3-5-sonnet-20240620-v1:0"

    native_request = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 512,
        "temperature": 0.5,
        "messages": [
            {
                "role": "user",
                "content": [{"type": "text", "text": prompt}],
            }
        ],
    }

    request = json.dumps(native_request)

    try:
        response = client.invoke_model(modelId=model_id, body=request)

    except (ClientError, Exception) as e:
        print(f"ERROR: Can't invoke '{model_id}'. Reason: {e}")
        exit(1)

    # Decode the response body.
    model_response = json.loads(response["body"].read())

    # Extract and print the response text.
    return model_response["content"][0]["text"]

In [ ]:
# Gradio works by defining a function that you wish to run in the interface
# Our file and prompt are inputs and it returns the llm response

def gen_insight(data, data_type):
    data_loaded = DataFrameLoader(data, page_content_column="product_category_name_english").load()
    prompt = "The following is the data on the " + data_type + " product categories. Highlight three insights from this data in bullet format. Data: "  + str(data_loaded)
    return get_output_aws(prompt)

def llm_query_df(month, prompt):
    product_data = pd.read_csv(DATA_FOLDER + "monthly_product_totals.csv")
    one_month_product_data = product_data[product_data["order_month"]==month]

    snip_1 = one_month_product_data.sort_values("total_price", ascending=False).head()
    snip_2 = one_month_product_data.sort_values("unique_orders", ascending=False).head()
    snip_3 = one_month_product_data.sort_values("MoM_total_price", ascending=False).head()
    snip_4 = one_month_product_data.sort_values("MoM_total_price", ascending=True).head()
    snip_5 = one_month_product_data.sort_values("YoY_total_price", ascending=False).head()
    snip_6 = one_month_product_data.sort_values("YoY_total_price", ascending=True).head()

    highest_earners = gen_insight(snip_1, "highest earning")
    most_orederd = gen_insight(snip_2, "most ordered")
    high_monthly_growth = gen_insight(snip_3, "fastest growing (monthly)")
    # low_monthly_growth = gen_insight(snip_4, "slowest growing (monthly)")
    high_yearly_growth = gen_insight(snip_5, "fastest growing (yearly)")
    # low_yearly_growth= gen_insight(snip_6,  "slowest growing (yearly)")

    return highest_earners, most_orederd, high_monthly_growth, high_yearly_growth

In [ ]:
# Create the Gradio interface
iface = gr.Interface(
    fn=llm_query_df,  # Function to be called
    inputs=[
        gr.Dropdown(choices=month_list, label="Select the relevent month..."),  # File dropdown
    ],
    outputs=[gr.Textbox(label="Highest Earning"), gr.Textbox(label="Most Ordered"), 
             gr.Textbox(label="Fastest Growing (monthly)"),  gr.Textbox(label="Fastest Growing (yearly)")],  # Output type
    title="Query your Data",  # Title of the app
    description="This is a description",  # Description of the app
)

iface.launch()

c:\Users\annak\OneDrive\Documents\GitHub\data_science_ai_workshop_series\.conda\Lib\site-packages\gradio\utils.py:1024: UserWarning: Expected 2 arguments for function <function llm_query_df at 0x000001EBC1C14F40>, received 1.
  warnings.warn(
c:\Users\annak\OneDrive\Documents\GitHub\data_science_ai_workshop_series\.conda\Lib\site-packages\gradio\utils.py:1028: UserWarning: Expected at least 2 arguments for function <function llm_query_df at 0x000001EBC1C14F40>, received 1.
  warnings.warn(


* Running on local URL:  http://127.0.0.1:7862

To create a public link, set `share=True` in `launch()`.


c:\Users\annak\OneDrive\Documents\GitHub\data_science_ai_workshop_series\.conda\Lib\site-packages\gradio\helpers.py:968: UserWarning: Unexpected argument. Filling with None.
  warnings.warn("Unexpected argument. Filling with None.")
c:\Users\annak\OneDrive\Documents\GitHub\data_science_ai_workshop_series\.conda\Lib\site-packages\gradio\blocks.py:1836: UserWarning: A function (llm_query_df) returned too many output values (needed: 4, returned: 6). Ignoring extra values.
    Output components:
        [textbox, textbox, textbox, textbox]
    Output values returned:
        ["Based on the provided data, here are three insights in bullet format:

• Furniture and decor is the highest-earning product category, with total sales of $13,036.63 in January 2017.

• Health and beauty products show the second-highest total sales at $12,366.44, despite having fewer unique orders (79) compared to furniture and decor (132).

• The garden tools category experienced the most significant month-over-month